# Notebook04 exact cold resume

This notebook reconnects to the active bounded experiment, or resumes its exact checkpoints in detached tmux if the worker has stopped and time remains. **Run All performs validation, launch/reconnect, monitoring and results display.** It preserves the original three-hour deadline and independent watchdog.

A serialization mismatch was found after registration: two engineering `weighted_losses` dictionaries originally had integer microbatch keys, while JSON loads them as strings. Sorted hashing orders those key types differently. This adapter restores only those key types and requires the resulting plan hash to equal the already registered hash exactly. Every JSON-observable value, arm configuration, source hash and checkpoint remains unchanged. It refuses any broader mismatch.

Inputs: `artifacts/notebook_04/semantic-followup-001/plan.json`, registration, frozen sources and full-state checkpoints. Outputs: a recovery identity certificate and, only when needed, a detached recovery worker notebook/log. Checkpoints remain under `checkpoints/semantic-followup-001/`. Validation takes seconds; remaining training/evaluation uses only time left before **2026-09-27 02:25:06 UTC**. Completed runs display evidence even after that deadline.

Exact resume: retain `RUN_ID`, restart the `Python (ahr-go)` kernel and Run All here. Resume cell: `launch-exact-followup-resume`. The original primary notebook remains an immutable record of the registered experiment.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat, importlib.util, sys
ROOT=Path("/home/benjamin/workspace/ahr_go/fmm_jeppa")
_boot={}
for _cell in nbformat.read(ROOT/"fmm_jepa/bootstrap.ipynb",as_version=4).cells:
    if _cell.cell_type=="code" and "ahr-export" in _cell.metadata.get("tags",[]):
        exec(compile(_cell.source,"bootstrap#ahr-export","exec"),_boot)
_boot["install_notebook_package"](ROOT)
_spec=importlib.util.spec_from_loader("fmm_followup_workflow",_boot["TaggedNotebookLoader"](ROOT/"experiments/04_followup/workflow.ipynb"))
WORKFLOW=importlib.util.module_from_spec(_spec);sys.modules[_spec.name]=WORKFLOW;_spec.loader.exec_module(WORKFLOW)


## Narrow identity-preserving recovery adapter

In [ ]:
from contextlib import contextmanager
from pathlib import Path
import copy, hashlib, importlib.util, json, os, shlex, subprocess, sys, time
import nbformat
from fmm_jepa.training import atomic_json, digest_file, run_writer
from fmm_jepa.food101_training import _hash

RECOVERY_NOTEBOOK = 'notebooks/04_Followup_Exact_Resume.ipynb'

def load_legacy_bound_plan(root, run_id):
    """Restore only JSON-erased integer metadata keys and prove exact identity.

    This does not relax the registration hash or alter any model, optimizer,
    schedule, data, source, seed, checkpoint, deadline or numeric value.
    """
    root=Path(root).resolve();artifact=root/'artifacts/notebook_04'/run_id
    path=artifact/'plan.json';disk=json.loads(path.read_text())
    registration=json.loads((artifact/'factorial_registration.json').read_text())
    restored=copy.deepcopy(disk);changes=[]
    models=restored['engineering_selection']['equivalence_certificate']['models']
    if set(models) != {'vit','semantic'}:raise ValueError('Unexpected engineering model metadata; manual audit required')
    for kind in ('vit','semantic'):
        losses=models[kind]['weighted_losses']
        if not losses or any(not isinstance(k,str) or not k.isdecimal() or str(int(k))!=k for k in losses):
            raise ValueError('Unexpected weighted-loss keys; refuse broad type coercion')
        normalized={int(k):v for k,v in losses.items()}
        if len(normalized)!=len(losses):raise ValueError('Integer conversion would merge metadata keys')
        if any(k not in (16,32,64,128) for k in normalized):raise ValueError('Unregistered microbatch metadata key')
        models[kind]['weighted_losses']=normalized
        changes.append({'path':'engineering_selection.equivalence_certificate.models.'+kind+'.weighted_losses',
                        'old_key_types':'str','restored_key_types':'int','keys':list(normalized)})
    if json.loads(json.dumps(restored)) != disk:
        raise AssertionError('Recovery changed a JSON-observable value')
    restored_hash=_hash(restored);registered_hash=registration['binding']['plan_hash']
    if restored_hash != registered_hash:
        raise ValueError('Narrow metadata repair does not reproduce the exact registered plan; resume refused')
    WORKFLOW.check_followup_plan(root,restored)
    if restored['run_id'] != run_id:raise ValueError('Run identity mismatch')
    for arm in restored['arms']:
        protocol_path=artifact/'arms'/arm['id']/'protocol.json'
        if digest_file(protocol_path)!=registration['arms'][arm['id']]['protocol_sha256']:
            raise ValueError('Arm protocol checksum changed')
        protocol=json.loads(protocol_path.read_text())
        if protocol['arm']!=arm or protocol['binding']!=registration['binding']:
            raise ValueError('Arm plan or binding differs from original registration')
    clock_path=artifact/'budget_clock.json';clock=json.loads(clock_path.read_text())
    if clock != restored['budget']['clock']:raise ValueError('Persisted deadline differs from registered clock')
    certificate={'schema':'notebook04-legacy-plan-key-recovery/v1','status':'PASS',
        'run_id':run_id,'original_disk_plan_sha256':digest_file(path),
        'canonical_disk_plan_hash':_hash(disk),'restored_plan_hash':restored_hash,
        'registered_plan_hash':registered_hash,'restored_metadata_key_types':changes,
        'json_roundtrip_values_identical':True,'all_arm_protocols_unchanged':True,
        'frozen_source_hashes_unchanged':True,'deadline_unix':clock['deadline_unix'],
        'clock_sha256':digest_file(clock_path),'model_or_training_value_changes':False,
        'recovery_notebook':RECOVERY_NOTEBOOK}
    return restored,certificate

@contextmanager
def restored_plan_reader(root, run_id):
    """Patch only the exact plan-path read in the frozen workflow; restore finally."""
    root=Path(root).resolve();target=(root/'artifacts/notebook_04'/run_id/'plan.json').resolve()
    original=WORKFLOW._json
    def read(path):
        if Path(path).resolve()==target:return load_legacy_bound_plan(root,run_id)[0]
        return original(path)
    WORKFLOW._json=read
    try:yield
    finally:WORKFLOW._json=original

def recovery_export_hash(root):
    nb=nbformat.read(Path(root)/RECOVERY_NOTEBOOK,as_version=4)
    sources=[c.source for c in nb.cells if c.cell_type=='code' and 'ahr-export' in c.metadata.get('tags',[])]
    return hashlib.sha256('\n'.join(sources).encode()).hexdigest()

def run_exact_resume_worker(root,run_id):
    if not os.environ.get('TMUX'):raise RuntimeError('Recovery worker must run in detached tmux')
    root=Path(root).resolve();artifact=root/'artifacts/notebook_04'/run_id
    plan,certificate=load_legacy_bound_plan(root,run_id)
    launch=json.loads((artifact/'exact_resume_launch.json').read_text())
    if launch['recovery_export_sha256']!=recovery_export_hash(root):raise ValueError('Recovery adapter changed after launch')
    if launch['registered_plan_hash']!=certificate['registered_plan_hash']:raise ValueError('Recovery registration changed')
    if time.time()>=certificate['deadline_unix']:raise TimeoutError('Original wall-clock deadline exhausted; no extension')
    with restored_plan_reader(root,run_id):
        return WORKFLOW.run_bounded_followup(root,run_id)

def launch_exact_resume(root,run_id):
    root=Path(root).resolve();artifact=root/'artifacts/notebook_04'/run_id
    plan,certificate=load_legacy_bound_plan(root,run_id)
    session='fmm-followup-'+run_id
    # Existing session remains the sole writer. Reconnecting starts no model work.
    if WORKFLOW.session_exists(session):
        WORKFLOW.load_followup(root,'deadline_watchdog').launch_deadline_watchdog(root,run_id)
        return {'status':'EXISTING_SESSION','session':session,'attach':'tmux attach -t '+session,
                'identity_recovery_verified':True,'deadline_unix':certificate['deadline_unix']}
    status_path=artifact/'pilot_status.json'
    if status_path.exists() and json.loads(status_path.read_text()).get('status')=='COMPLETED':
        return {'status':'COMPLETED','artifact':str(artifact),'identity_recovery_verified':True}
    if time.time()>=certificate['deadline_unix']:raise TimeoutError('Original wall-clock deadline exhausted; saved evidence retained')
    with run_writer(artifact):
        current=nbformat.read(root/RECOVERY_NOTEBOOK,as_version=4)
        setup=next(c.source for c in current.cells if c.get('id')=='bootstrap-exact-resume')
        code=setup+'\n'
        code+='RECOVERY_PATH=ROOT/'+repr(RECOVERY_NOTEBOOK)+'\n'
        code+='RECOVERY_SPEC=importlib.util.spec_from_loader("fmm_followup_exact_recovery",_boot["TaggedNotebookLoader"](RECOVERY_PATH))\n'
        code+='RECOVERY=importlib.util.module_from_spec(RECOVERY_SPEC);RECOVERY.WORKFLOW=WORKFLOW;sys.modules[RECOVERY_SPEC.name]=RECOVERY;RECOVERY_SPEC.loader.exec_module(RECOVERY)\n'
        code+='RECOVERY.run_exact_resume_worker(ROOT,'+repr(run_id)+')\n'
        worker=artifact/'exact_resume_worker.ipynb'
        if worker.exists():
            if [c.source for c in nbformat.read(worker,as_version=4).cells if c.cell_type=='code']!=[code]:
                raise ValueError('Existing exact-resume worker code differs')
        else:
            notebook=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Exact Notebook04 cold-resume worker\nOriginal plan values, full-state checkpoints and deadline retained. The recovery adapter only restores integer keys lost in engineering metadata JSON.'),nbformat.v4.new_code_cell(code)],
                metadata={'kernelspec':{'display_name':'Python (ahr-go)','language':'python','name':'ahr-go'}})
            nbformat.write(notebook,worker)
        command=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
                 '--ExecutePreprocessor.timeout=-1','--ExecutePreprocessor.kernel_name=ahr-go',str(worker)]
        launch={'session':session,'command':command,'attach':'tmux attach -t '+session,'detach':'Ctrl-b, then d',
                'deadline_unix':certificate['deadline_unix'],'registered_plan_hash':certificate['registered_plan_hash'],
                'recovery_export_sha256':recovery_export_hash(root),'worker':str(worker),'log':str(artifact/'exact_resume_worker.log')}
        atomic_json(artifact/'exact_resume_launch.json',launch)
        atomic_json(artifact/'exact_resume_identity_certificate.json',certificate)
        shell='exec '+shlex.join(command)+' > '+shlex.quote(str(artifact/'exact_resume_worker.log'))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)
    WORKFLOW.load_followup(root,'deadline_watchdog').launch_deadline_watchdog(root,run_id)
    return {'status':'LAUNCHED',**launch}


## Validate the cold-resume identity before any model work

In [ ]:
RUN_ID='semantic-followup-001'
RESTORED_PLAN, RECOVERY_CERTIFICATE=load_legacy_bound_plan(ROOT,RUN_ID)
# Read-only cold-path validation: force the frozen runner to verify the restored
# binding in a fresh module. No model construction, resume, optimizer or GPU work.
RUNNER=WORKFLOW.load_followup(ROOT,'runner')
BUNDLE=WORKFLOW.reuse_food_bundle(ROOT,RESTORED_PLAN['arms'][0]['config'],verify_images=False)
REGISTRATION=RUNNER._registered(ROOT,ROOT/'artifacts/notebook_04'/RUN_ID,RESTORED_PLAN,BUNDLE)
# Existing-certificate branch validates only hashes/manifests and cannot train.
assert (ROOT/'artifacts/notebook_04'/RUN_ID/'factorial_preflight.json').is_file()
REUSED_PREFLIGHT=RUNNER.verify_factorial_resume(ROOT,ROOT/'artifacts/notebook_04'/RUN_ID,RESTORED_PLAN,BUNDLE,RECOVERY_CERTIFICATE['deadline_unix'])
assert REUSED_PREFLIGHT['status']=='PASS'
RECOVERY_CERTIFICATE['original_full_state_preflight_reuse']='PASS'
_original_reader=WORKFLOW._json
with restored_plan_reader(ROOT,RUN_ID):
    _reloaded=WORKFLOW._json(ROOT/'artifacts/notebook_04'/RUN_ID/'plan.json')
    assert _hash(_reloaded)==RECOVERY_CERTIFICATE['registered_plan_hash']
    assert WORKFLOW._json(ROOT/'artifacts/notebook_04'/RUN_ID/'budget_clock.json')==RESTORED_PLAN['budget']['clock']
assert WORKFLOW._json is _original_reader
try:
    with restored_plan_reader(ROOT,RUN_ID):
        raise RuntimeError('intentional CPU reader-restoration test')
except RuntimeError as _error:
    assert str(_error)=='intentional CPU reader-restoration test'
assert WORKFLOW._json is _original_reader
# Validate the exact tagged-module import path used by a future detached worker.
_recovery_spec=importlib.util.spec_from_loader('fmm_followup_exact_recovery_cpu_check',_boot['TaggedNotebookLoader'](ROOT/RECOVERY_NOTEBOOK))
_recovery_module=importlib.util.module_from_spec(_recovery_spec);_recovery_module.WORKFLOW=WORKFLOW
sys.modules[_recovery_spec.name]=_recovery_module;_recovery_spec.loader.exec_module(_recovery_module)
assert _recovery_module.load_legacy_bound_plan(ROOT,RUN_ID)[1]['registered_plan_hash']==RECOVERY_CERTIFICATE['registered_plan_hash']
RECOVERY_CERTIFICATE['tagged_worker_module_import']='PASS'
RECOVERY_CERTIFICATE['frozen_runner_cold_binding_check']='PASS'
RECOVERY_CERTIFICATE['scoped_reader_restoration_check']='PASS'
RECOVERY_CERTIFICATE['validation_started_training']=False
atomic_json(ROOT/'artifacts/notebook_04'/RUN_ID/'exact_resume_identity_certificate.json',RECOVERY_CERTIFICATE)
RECOVERY_CERTIFICATE


## Reconnect or resume within the original deadline

In [ ]:
# The frozen CSV writer expects its output directory to exist.
_RECOVERY_ARTIFACT=ROOT/'artifacts/notebook_04'/RUN_ID
(_RECOVERY_ARTIFACT/'predictions').mkdir(parents=True,exist_ok=True)
_LAUNCH_NOT_BEFORE=time.time()
LAUNCH=launch_exact_resume(ROOT,RUN_ID)
LAUNCH


In [ ]:
# A newly launched worker may briefly coexist with the prior FAILED status.
# Require a fresh status from this launch before handing control to the frozen
# monitor. A fresh failure remains an error; old status is never overwritten.
if LAUNCH['status']=='LAUNCHED':
    _startup_deadline=min(_LAUNCH_NOT_BEFORE+55.0,float(RECOVERY_CERTIFICATE['deadline_unix']))
    _status_path=_RECOVERY_ARTIFACT/'pilot_status.json'
    while True:
        _status=json.loads(_status_path.read_text()) if _status_path.exists() else {}
        if float(_status.get('updated_unix',0))>=_LAUNCH_NOT_BEFORE:
            break
        if time.time()>=_startup_deadline:
            raise TimeoutError('No fresh worker status within the bounded startup wait. Inspect exact_resume_worker.log; the original deadline remains in force.')
        if not WORKFLOW.session_exists(LAUNCH['session']):
            raise RuntimeError('Recovery worker exited before publishing a fresh status. Inspect exact_resume_worker.log; saved checkpoints remain intact.')
        time.sleep(min(1.0,max(0.0,_startup_deadline-time.time())))
COMPLETION=WORKFLOW.monitor_bounded_followup(ROOT,RUN_ID)
COMPLETION


## Saved results

In [ ]:
from IPython.display import display,Markdown,Image
ARTIFACT=ROOT/'artifacts/notebook_04'/RUN_ID
for path in sorted(ARTIFACT.glob('*results*.md')):display(Markdown(path.read_text()))
for path in sorted(ARTIFACT.glob('factorial_*.png')):display(Image(filename=str(path)))